# 2. Named Entity Recognition (Section 4.1, Table 2)

spaCy (`pt_core_news_sm`) and XLM-RoBERTa (`Davlan/xlm-roberta-base-ner-hrl`) are applied to the
**preprocessed** text, as in the paper. In the paper run, FW.Br texts were truncated to 1,000
characters for XLM-RoBERTa and FT.Br texts were not; this is reproduced below.

Requires `python -m spacy download pt_core_news_sm`. A GPU makes XLM-RoBERTa much faster.

In [ ]:
from utils import *
import spacy, torch
from transformers import pipeline

In [ ]:
nlp = spacy.load("pt_core_news_sm")
ner = pipeline("ner", model="Davlan/xlm-roberta-base-ner-hrl", aggregation_strategy="simple",
               device=0 if torch.cuda.is_available() else -1)
MAX_CHARS = {"whatsapp": 1000, "telegram": None}

In [ ]:
rows = []
for platform in ["whatsapp", "telegram"]:
    texts = load_dataset(platform)["text_processed"].tolist()

    spacy_counter = Counter(ent.text for doc in nlp.pipe(texts) for ent in doc.ents)

    xlm_counter = Counter()
    for text in texts:
        if text.strip():
            text = text[:MAX_CHARS[platform]] if MAX_CHARS[platform] else text
            xlm_counter.update(ent["word"] for ent in ner(text))

    for model, counter in [("spaCy", spacy_counter), ("XLM-RoBERTa", xlm_counter)]:
        for rank, (entity, freq) in enumerate(counter.most_common(10), 1):
            rows.append({"dataset": NAMES[platform], "model": model, "rank": rank,
                         "entity": entity, "frequency": freq})

entities = pd.DataFrame(rows)
entities.to_csv(results_folder("02_named_entities") / "top_entities.csv", index=False)
entities.pivot_table(index="rank", columns=["dataset", "model"], values="entity", aggfunc="first")

## Comparison with the paper (Table 2, original Portuguese terms)

In [ ]:
paper = {
    ("FW.Br", "spaCy"): "jesus, brasil, jesus cristo, deus, ifood, chirley, hostilio souza, misericordia, acredito, familia",
    ("FW.Br", "XLM-RoBERTa"): "brasil, je, jesus, sus, israel, ley, galile, jesus cristo, reginaldo manzotti",
    ("FT.Br", "spaCy"): "brasil, jesus, jesus cristo, reino unido, deus, jesus salva leia biblia, eua, brazil, venezuela, edson",
    ("FT.Br", "XLM-RoBERTa"): "brasil, jesus, je, sus, da, brazil, israel, mari, deus",
}
for (dataset, model), expected in paper.items():
    got = entities.query("dataset == @dataset and model == @model")["entity"].tolist()
    missing = [e for e in expected.split(", ") if e not in got]
    print(f"{dataset} {model}: {'all paper entities found' if not missing else 'missing: ' + str(missing)}")